In [1]:
# try to get a simple naive forecast as baseline prediction for electricity demand in future

import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, root_mean_squared_error

In [2]:

# aim is to predict electricity demand 24 hours in advance


# naive models to test:

# - just the last observed value (previous hour)
# - same datetime previous day
# - same datetime previous week
# - a window average - use average of the previous week, past month
# - Average of that datetime across all other weeks of dataset


# loss to use: for now use default of MSE
# no need to worry about data leakage etc. with the naive model?

df = pd.read_csv('data/processed/hourly_usage_cleaned.csv')

# drop the unnamed column
df.drop(columns=['Unnamed: 0'], inplace=True)

In [3]:
# try setting datetime and client as index
df_multiindex = df.set_index(['datetime', 'client_id'])

df_multiindex

,,hourly_usage_kwh
datetime,client_id,
2012-01-01 01:00:00,1,4.124365
2012-01-01 02:00:00,1,4.758883
2012-01-01 03:00:00,1,4.124365
2012-01-01 04:00:00,1,4.758883
2012-01-01 05:00:00,1,4.441624
...,...,...
2014-12-31 20:00:00,370,8729.729730
2014-12-31 21:00:00,370,8216.216216
2014-12-31 22:00:00,370,8297.297297


In [11]:
# # get the naive forecast for each hour (hourly usage is the target)

# the first naive prediction is just 24 hours prior lag
df_multiindex['yn_1_day_lag'] = df_multiindex.groupby(level='client_id')['hourly_usage_kwh'].shift(24)

# then naive prediciton of one week lag
df_multiindex['yn_1_wk_lag'] = df_multiindex.groupby(level='client_id')['hourly_usage_kwh'].shift(7*24)

# variable for the datetime index for cleaner coding
datetime_index =  pd.to_datetime(df_multiindex.index.get_level_values('datetime'))

# then use mean of that day of week, time and month across all points in dataset
df_multiindex['yn_avg_time_day_month'] =  df_multiindex.groupby([pd.Grouper(level='client_id'),
                                                                 datetime_index.hour,
                                                                 datetime_index.dayofweek,
                                                                 datetime_index.month])['hourly_usage_kwh'].transform('mean')

# drop nan rows (because data where not possible to predict the target can be omitted)
df_multiindex.dropna(inplace=True)

# check that clients have nans where should
df_multiindex.xs(4, level='client_id').head(50)

,hourly_usage_kwh,yn_1_day_lag,yn_1_wk_lag,yn_avg_time_day_month
datetime,,,,
2012-01-09 01:00:00,105.182927,152.439024,109.756098,117.026266
2012-01-09 02:00:00,86.382114,115.345528,97.560976,97.912758
2012-01-09 03:00:00,86.890244,100.101626,95.020325,92.362414
2012-01-09 04:00:00,55.894309,71.138211,65.548780,76.923077
2012-01-09 05:00:00,53.353659,60.467480,61.483740,72.232645
2012-01-09 06:00:00,64.024390,71.138211,68.089431,74.812383
2012-01-09 07:00:00,83.333333,88.414634,94.512195,89.509068
2012-01-09 08:00:00,102.134146,101.626016,106.199187,108.388055
2012-01-09 09:00:00,102.134146,106.199187,104.674797,104.713884


In [12]:
df_multiindex.columns

Index(['hourly_usage_kwh', 'yn_1_day_lag', 'yn_1_wk_lag',
       'yn_avg_time_day_month'],
      dtype='object')

In [13]:
# and now to compute the loss of naive predictions vs the target. For now just use the MSE

#TODO: should pass in each individual client seperately 

# # first get name of columns that are the naive predictions (i.e., all but target)
# naive_pred_cols = df_multiindex.columns[df_multiindex.columns != 'y']

# # Then dictionary containing the MSE values
# naive_forecast_mse_dict = {naive_pred_name: mean_squared_error(y_true=df_multiindex['y'], y_pred=df_multiindex[naive_pred_name]) 
#                            for naive_pred_name in naive_pred_cols}

# naive_forecast_rmse_dict = {naive_pred_name: root_mean_squared_error(y_true=df_multiindex['y'], y_pred=df_multiindex[naive_pred_name]) 
#                            for naive_pred_name in naive_pred_cols}

# get rmse dictionary containing rmse of different naive predictions for each client
predictions = df_multiindex.columns[df_multiindex.columns != 'hourly_usage_kwh']

rmse_dict = {col: df_multiindex.groupby(level='client_id').apply(
    lambda g: root_mean_squared_error(y_true=g['hourly_usage_kwh'], y_pred=g[f'{col}'])) for col in predictions}

# # try compute rmse for each client
# rmse_each_client = df_multiindex.groupby(level='client_id').apply(
#     lambda g: root_mean_squared_error(y_true=g['hourly_usage_kwh'], y_pred=g['yn_1_day_lag']))

# # and compute the mean squared error for each client
# mean_usage_each_client = df_multiindex.groupby(level='client_id').apply(
#     lambda g: mean_squared_error(y_true=g['hourly_usage_kwh'], y_pred=g['yn_1_day_lag']))


In [14]:
# check rmse weighted y the mean of usage of each client
mean_usage_each_client = df_multiindex.groupby(level='client_id')['hourly_usage_kwh'].mean()

In [19]:
nrmse_1week = rmse_dict['yn_1_wk_lag']/mean_usage_each_client
nrmse_1day = rmse_dict['yn_1_day_lag']/mean_usage_each_client
nrmse_1avg = rmse_dict['yn_avg_time_day_month']/mean_usage_each_client


print(nrmse_1week[nrmse_1week < 0.1].shape, '\n')
print(nrmse_1day[nrmse_1day < 0.1].shape, '\n')
print(nrmse_1avg[nrmse_1avg < 0.1].shape, '\n')

nrmse_1week_mean = nrmse_1week.mean()
nrmse_1day_mean = nrmse_1day.mean()
nrmse_1avg_mean = nrmse_1avg.mean()

print(nrmse_1week_mean, '\n')
print(nrmse_1day_mean, '\n')
print(nrmse_1avg_mean, '\n')

(51,) 

(100,) 

(46,) 

0.19071813396120596 

0.19750221376834784 

0.1849622204294119 



client_id
3      2.682209
93     2.074675
94     1.094723
130    1.666578
131    1.335810
132    1.279584
347    1.094126
348    1.254947
366    1.004495
dtype: float64